In [1]:
! pip install chronos-forecasting matplotlib # install the chronos model

In [2]:
import torch
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt


In [3]:
# load the file directly from Huggingface

url = ("https://huggingface.co/datasets/Tristanchou/"
       "Preference-Conditioned-Heterogeneous-MARL-Microgrid/resolve/main/microgrid_opsd_multiyear.csv")

# Parse timestamp to convert it to computer-readable date-time objects
df = pd.read_csv(url, parse_dates=["timestamp"]) 

print(f"The dataset shape: {df.shape}")
print( df.head())

The dataset shape: (50401, 19)
                  timestamp     pv_kw    wind_kw     load_kw  \
0 2014-12-31 23:00:00+00:00  0.202918  39.334556  186.892131   
1 2015-01-01 00:00:00+00:00  0.202918  39.334556  186.892131   
2 2015-01-01 01:00:00+00:00  0.202918  39.273729  187.654492   
3 2015-01-01 02:00:00+00:00  0.202918  39.687594  172.550697   
4 2015-01-01 03:00:00+00:00  0.202918  42.085626  164.765285   

   critical_load_kw  flexible_load_kw  price_cny_kwh  carbon_factor_kg_kwh  \
0        117.742043         69.150088       0.217932              0.713735   
1        117.742043         69.150088       0.217932              0.713735   
2        118.222330         69.432162       0.217932              0.714557   
3        108.706939         63.843758       0.217932              0.699236   
4        103.802130         60.963155       0.217932              0.685588   

   risk_level  reserve_ratio  island_flag  hour  month  dayofyear  \
0    0.551560            0.0            0    2

In [4]:
# let's see what the target looks like
print(df["pv_kw"].describe())

count    50401.000000
mean        26.960887
std         38.735809
min          0.000000
25%          0.000000
50%          2.685499
75%         46.183642
max        185.131766
Name: pv_kw, dtype: float64


In [5]:
# let's confirm that there are no missing hours

pv = df.set_index("timestamp")["pv_kw"]["2018-01-01":]   # about 2.7 years
print(pv.index.to_series().diff().value_counts().head())  # should be all 1 hour
train, test = pv[:-168], pv[-168:]                        # last 7 days held out

timestamp
0 days 01:00:00    24095
Name: count, dtype: int64


In [6]:
# Let's try some zero-shot forecasting first with Chronos T5 small


from chronos import ChronosPipeline
pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-small",
                                       device_map="cuda", dtype=torch.float32)
ctx = torch.tensor(train.values[-512:], dtype= torch.float32)
fc = pipe.predict(ctx, prediction_length=24, num_samples=20)   # [1, 20, 24]
low, med, high = np.quantile(fc[0].numpy(), [0.1, 0.5, 0.9], axis=0)

config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  185MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [7]:
naive = train.values[-24:]                    # "tomorrow looks like today"
mae = lambda a, b: np.mean(np.abs(a - b))
print(mae(med, test.values[:24]), mae(naive, test.values[:24]))

2.6730854415291296 6.329666458333334


In [8]:
def multidays_mae(pipe, series, n_days=30):
    errs = []
    for d in range(n_days):
        end = len(series) - (n_days - d) * 24        # where this forecast day starts
        ctx = torch.tensor(series.values[end-512:end], dtype=torch.float32)
        fc = pipe.predict(ctx, prediction_length=24, num_samples=20)
        med = np.quantile(fc[0].numpy(), 0.5, axis=0)
        actual = series.values[end:end+24]
        base = series.values[end-24:end]             # "same as yesterday"
        errs.append((np.mean(np.abs(med - actual)), np.mean(np.abs(base - actual))))
    return np.mean(errs, axis=0)

print(multidays_mae(pipe, pv, 30))

# The scores are chronos scores and the baseline scores before fine-tuning

[5.6419349  6.25113925]


In [9]:
# To begin forecasting, we convert the dataset to a format suitable for teh Chronos model. 
# See scripts/README.md in the repo

!pip install gluonts pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.0/864.0 kB 22.9 MB/s  0:00:00


In [10]:
from gluonts.dataset.arrow import ArrowWriter

full = df.set_index("timestamp")["pv_kw"]

practice = full.values[:-720].astype("float32")   # everything minus the last 720 hours (30 days) 

data = [
    {"start": np.datetime64("2014-12-31 23:00", "s"), "target": practice}
    ]

ArrowWriter(compression="lz4").write_to_file(data, path="/kaggle/working/train.arrow")
print(len(practice))

49681


The above is accurate because 50401-720 = 49681

In [11]:
!pip install "chronos-forecasting[training] @ git+https://github.com/amazon-science/chronos-forecasting.git"
!git clone https://github.com/amazon-science/chronos-forecasting.git

# To see the config files
!ls chronos-forecasting/scripts/training/configs

  Cloning https://github.com/amazon-science/chronos-forecasting.git to /tmp/pip-install-f9l0pr9i/chronos-forecasting_de6830bd8eab440b8387e6a9ccc2ca74
  Running command git clone --filter=blob:none --quiet https://github.com/amazon-science/chronos-forecasting.git /tmp/pip-install-f9l0pr9i/chronos-forecasting_de6830bd8eab440b8387e6a9ccc2ca74
  Resolved https://github.com/amazon-science/chronos-forecasting.git to commit 1224fafa91e1ecd6674f5204066b159e40b5e4a1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for chronos-forecasting: filename=chronos_forecasting-2.3.3.dev0-py3-none-any.whl size=81051 sha256=ff00ff57e72c81dfc96fc03ad6a101a06be99227bc47ac709d3a81a6c46280c8
  Stored in directory: /tmp/pip-ephem-wheel-cache-o5to1y7v/wheels/15/c5/bc/ac4e8cdaf8a0432d6323adb552caa3eab6551c8338ba45f1f9
Successfully built chronos-forecasting
  Attempting uninstall: chronos-forecasting
    Found exi

In [12]:
import yaml
path = "chronos-forecasting/scripts/training/configs/chronos-t5-small.yaml"

cfg = yaml.safe_load(open(path))
cfg["training_data_paths"] = ["/kaggle/working/train.arrow"]   # our practice file

cfg["probability"] = [1.0]                                     # use it 100%

cfg["torch_compile"] = False                                   # avoids a slow start

yaml.safe_dump(cfg, open("/kaggle/working/my_config.yaml", "w"))
print(cfg)

{'training_data_paths': ['/kaggle/working/train.arrow'], 'probability': [1.0], 'context_length': 512, 'prediction_length': 64, 'min_past': 60, 'max_steps': 200000, 'save_steps': 100000, 'log_steps': 500, 'per_device_train_batch_size': 32, 'learning_rate': 0.001, 'optim': 'adamw_torch_fused', 'num_samples': 20, 'shuffle_buffer_length': 100000, 'gradient_accumulation_steps': 1, 'model_id': 'google/t5-efficient-small', 'model_type': 'seq2seq', 'random_init': True, 'tie_embeddings': True, 'output_dir': './output/', 'tf32': True, 'torch_compile': False, 'tokenizer_class': 'MeanScaleUniformBins', 'tokenizer_kwargs': {'low_limit': -15.0, 'high_limit': 15.0}, 'n_tokens': 4096, 'lr_scheduler_type': 'linear', 'warmup_ratio': 0.0, 'dataloader_num_workers': 1, 'max_missing_prop': 0.9, 'use_eos_token': True}


In [13]:
# some of the values are not in line with our proposed fine-tuning. For example:
# 'model_id': 'google/t5-efficient-small'. So we adjust those;
cfg["model_id"] = "amazon/chronos-t5-small"
cfg["random_init"] = False
cfg["max_steps"] = 1000
cfg["save_steps"] = 500
cfg["log_steps"] = 50
cfg["tf32"] = False
cfg["shuffle_buffer_length"] = 10000
cfg["output_dir"] = "/kaggle/working/output/"
yaml.safe_dump(cfg, open("/kaggle/working/my_config.yaml", "w"))

In [14]:
!pip install typer typer-config gluonts accelerate

In [15]:
# now, we fine-tune

# %cd chronos-forecasting/scripts
!CUDA_VISIBLE_DEVICES=0 python training/train.py --config /kaggle/working/my_config.yaml \
    --model-id amazon/chronos-t5-small \
    --no-random-init \
    --max-steps 1000 \
    --learning-rate 0.001

python3: can't open file '/kaggle/working/training/train.py': [Errno 2] No such file or directory


In [16]:
!ls /kaggle/working/output/run-0

# Now we have a fine-tuned model
# checkpoint-500: a saved copy of the model halfway through (step 500).
# checkpoint-1000: a copy at step 1,000, the end of training.
# checkpoint-final: the finished model, which should be the same as the one at step 1,000.

ls: cannot access '/kaggle/working/output/run-0': No such file or directory


In [17]:
# Now, let's test the model on the last 720 hours (30 days)
# Let's also use the halfway copy to see if more practice actually helped.

from chronos import ChronosPipeline
for name in ["checkpoint-500", "checkpoint-final"]:
    p = ChronosPipeline.from_pretrained(f"/kaggle/working/output/run-0/{name}",
                                        device_map="cuda", dtype=torch.float32)
    print(name, multidays_mae(p, pv, 30))

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/working/output/run-0/checkpoint-500'. Use `repo_type` argument if needed.

SO we see that the model improved with more practice, though not by a lot.  
It went from 5.72 (zer-shot) to 5.56 at the halfway point and to 5.39 at the end  

Some other things to not:  
1. All the test results come from the same 30 days, so a different month might give a bigger or smaller gain.
2. As mentioned in the dataset card on Huggingface, pv_kw is a rescaled, derived series, so the kW numbers in this dataset are synthetic, and not from a real plant.


**In conclusion, fine-tuning reduced the 30-day forecast error from 5.72 to 5.39 kW (about 6%), on a single held-out window and a single training run**